# Forecast labels and MAP baseline

Creates onset, pressure-drop, and burden labels from numeric arterial MAP. Patient partitions remain fixed before model fitting.

Run the setup cell first. Full processing and training require locally provisioned datasets; the quick start uses synthetic tests and saved aggregate results.


In [ ]:
# Load the libraries and project settings needed for this step.
from pathlib import Path
import sys, json, os, time, shutil
# Find the project from its configuration file rather than a machine-specific path.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs' / 'hemodynamics.json').exists())
assert Path(sys.executable).resolve() == (ROOT / '.venv' / 'Scripts' / 'python.exe').resolve(), 'Select the project .venv kernel'
sys.path.insert(0, str(ROOT / 'notebooks'))
import importlib, psutil
import pandas as pd
from hemodynamics import snapshot, labels
for _module in (snapshot, labels):   # pick up edited code in a long-lived kernel
    importlib.reload(_module)

# Check for competing processing jobs before starting a costly data pass.
me = psutil.Process()
for p in psutil.process_iter():
    try: p.cpu_percent(None)
    except psutil.Error: pass
time.sleep(0.8)
busy = []
for p in psutil.process_iter(['pid', 'name']):
    try:
        if p.pid != me.pid and p.info['name'] and 'python' in p.info['name'].lower() and p.cpu_percent(None) > 50:
            busy.append(p.pid)
    except psutil.Error:
        pass
# Inspect run locks and processing settings before launching this stage.
locks = [str(p.relative_to(ROOT)) for p in (ROOT / 'data' / 'checkpoints').rglob('*.lock')]
print('Root:', ROOT)
print(f'Free disk: {shutil.disk_usage(ROOT).free / 1e9:.0f} GB | free RAM: {psutil.virtual_memory().available / 1e9:.0f} GB')
print('Other busy python processes:', busy or 'none', '| run locks:', locks or 'none')
if busy or locks:
    print('WARNING: another pipeline looks active; this notebook is light, but wait if it is a long MOVER/VitalDB run.')

## 1. Back up and tag dataset v1
Idempotent: rerunning does nothing while the dataset is unchanged. Set `EXTRA_COPY` (for example `r'G:\backups'`) to also copy the backup to a second drive; it is hash-verified. The backup is about 75 MB (it does **not** include the 85 GB of case parquet files).

In [ ]:
EXTRA_COPY = None   # e.g. r'G:\backups'  (a folder on another drive); None = no second copy
tag = snapshot.snapshot_dataset_v1(ROOT, extra_dest=EXTRA_COPY)
display(pd.Series({
    'dataset version': tag['version'], 'reused existing snapshot': tag['reused'], 'snapshot folder': tag['snapshot_dir'],
    'files / MB': f"{tag['snapshot_files']} / {tag['snapshot_bytes'] / 1e6:.0f}", 'second copy': tag['extra_dest'],
    'cases': tag['n_cases'], 'patients': tag['n_patients'], 'cases by dataset': tag['cases_by_dataset'],
    'index content sha256': tag['index_content_sha256'][:16] + '...', 'frozen MOVER patient list sha256': str(tag['mover_patient_list_sha256'])[:16] + '...',
    'MOVER eligibility rung': tag['mover_rung']}, name='dataset_v1').to_frame())
print('Guard check (raises if the dataset changed after tagging):', snapshot.assert_dataset_v1(ROOT)['version'])

## 2. Label definitions (fixed and hashed)
Every output records `label_rule_sha256`. Key points: an event is a run of **consecutive MAP < 65 mmHg lasting >= 60 s** (sample gaps up to 10 s tolerated) that **starts within the 5-minute horizon** and whose qualifying minute uses only samples inside the horizon; windows with any MAP < 65 in the **last 60 s** are excluded as already hypotensive; dMAP and AUT65 follow the proposal's formulas (AUT65 in mmHg*min, exact piecewise-linear integral); negatives whose horizon has a >60 s MAP gap are flagged `label_ambiguous`.

In [ ]:
# Load the recorded settings or metadata for this processing step.
rule = labels.load_label_rule(json.loads((ROOT / 'configs' / 'hemodynamics.json').read_text()))
print(json.dumps(rule, indent=2)); print('label_rule_sha256:', labels.label_rule_sha256(rule))
print(labels.__doc__)

## 3. Generate labels
About 20 seconds on this machine (16 worker processes, one task per case). Rerunning skips cases whose MAP data, window list and rule are unchanged.

In [ ]:
# Run the configured processing step and retain its returned summary.
summary = labels.build_labels(ROOT, progress=True)
display(pd.DataFrame(summary['datasets']).T)

## 4. Label tables
`eligible` = ready windows with `min_valid_frac >= 0.7` that are not already hypotensive. Prevalence is the share of eligible windows with an event. MOVER's event rate is about twice VitalDB's, so expect calibration to shift across datasets; AUT65 and dMAP are zero-heavy (see the zero percentages), which is why the proposal allows a two-part regression for AUT65.

In [ ]:
# Inspect the selected settings and outputs for this notebook step.
by_split, pooled = labels.summarize_labels(ROOT, min_valid_frac=0.7)
display(by_split); display(pooled)

## 5. Baseline A: current MAP and short trend
Logistic regression on `map_current, map_mean_60, map_slope_60, map_min_60, map_std_60` (standardized), trained on train windows, evaluated on eligible, unambiguous windows with `min_valid_frac >= 0.7`. Confidence intervals resample **patients** (200 repetitions). `E3x` trains on VitalDB-train and tests on **all** MOVER patients (the strongest generalization test). The 'negative current MAP' column is the trivial score -MAP.

In [ ]:
# Inspect the selected settings and outputs for this notebook step.
baseline = labels.baseline_a(ROOT, reps=200)
rows = []
for name, e in baseline['settings'].items():
    m, ci = e['model'], e['model_ci95']
    rows.append({'setting': name, 'test patients': e['test_patients'], 'test windows': e['test_windows'], 'prevalence %': round(100 * e['prevalence'], 2),
                 'AUROC': f"{m['auroc']:.3f} [{ci['auroc'][0]:.3f}, {ci['auroc'][1]:.3f}]", 'AUPRC': f"{m['auprc']:.3f} [{ci['auprc'][0]:.3f}, {ci['auprc'][1]:.3f}]",
                 'Brier': round(m['brier'], 4), 'calibration slope': round(m['cal_slope'], 2), 'AUROC of -MAP only': round(e['negative_current_map_auroc'], 3),
                 'gray-zone AUROC': round(e['gray_zone']['auroc'], 3) if 'gray_zone' in e else None})
display(pd.DataFrame(rows))
pooled_ds = baseline['settings']['pooled_train->pooled_test'].get('by_dataset', {})
display(pd.DataFrame({k: {'patients': v['patients'], 'AUROC': round(v['auroc'], 3), 'AUROC 95% CI': [round(x, 3) for x in v['auroc_ci95']]} for k, v in pooled_ds.items()}).T)
display(pd.DataFrame(baseline['regression_references_pooled_test']).drop(index='test_windows', errors='ignore').T)

In [ ]:
# Load the libraries and project settings needed for this step.
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, setting in zip(axes, ('pooled_train->pooled_test', 'E3x vitaldb_train->all_mover')):
    # Label the panel with the measurement names and their units.
    c = labels.calibration_points(ROOT, setting)
    ax.plot([0, c.mean_predicted.max()], [0, c.mean_predicted.max()], 'k--', lw=1)
    ax.plot(c.mean_predicted, c.observed_rate, 'o-'); ax.set_title(setting, fontsize=9)
    ax.set_xlabel('mean predicted risk'); ax.set_ylabel('observed event rate')
plt.tight_layout(); plt.show()

## 6. Verify outputs
All checks should print `True`. Send back the two tables above if anything is `False`.

In [ ]:
# Summarize the records separately within each requested group.
lab_files = {d: labels.labels_path(ROOT, d) for d in ('mover', 'vitaldb')}
manifest = json.loads((ROOT / 'data' / 'manifests' / 'labels_manifest.json').read_text())
# Summarize the records separately within each requested group.
tag_now = snapshot.assert_dataset_v1(ROOT)
all_labels = labels.load_labels(ROOT, columns=['dataset', 'case_id', 'label_rule_sha256', 'dataset_version'])
index = pd.read_parquet(ROOT / 'data' / 'manifests' / 'unified_case_index.parquet')
# Summarize the records separately within each requested group.
checks = {
    'label files exist': all(p.exists() for p in lab_files.values()),
    'baseline_A.json + label_summary.json exist': all((ROOT / 'results' / f).exists() for f in ('baseline_A.json', 'label_summary.json')),
    'labels carry the current rule hash': set(all_labels.label_rule_sha256) == {labels.label_rule_sha256(rule)},
    'labels tied to dataset v1': set(all_labels.dataset_version) == {tag_now['version']} and manifest['dataset_index_sha256'] == tag_now['index_content_sha256'],
    'cases with labels / cases in index': f"{all_labels.groupby('dataset').case_id.nunique().to_dict()} / {index.groupby('dataset').size().to_dict()}",
}
for k, v in checks.items(): print(f'{k}: {v}')